In [4]:
from pyspark.sql import SparkSession
from pyspark.ml.feature import VectorAssembler, StringIndexer
from pyspark.ml.classification import LogisticRegression
from pyspark.ml import Pipeline

# Create Spark session
spark = SparkSession.builder.appName("FlightDelayPrediction").getOrCreate()

# Load preprocessed data
df = spark.read.csv("preprocessed_version.csv", header=True, inferSchema=True)

# Show schema to confirm
df.printSchema()

root
 |-- FL_DATE: date (nullable = true)
 |-- OP_CARRIER: string (nullable = true)
 |-- OP_CARRIER_FL_NUM: integer (nullable = true)
 |-- ORIGIN: string (nullable = true)
 |-- DEST: string (nullable = true)
 |-- CRS_DEP_TIME: double (nullable = true)
 |-- DEP_TIME: double (nullable = true)
 |-- DEP_DELAY: double (nullable = true)
 |-- TAXI_OUT: double (nullable = true)
 |-- WHEELS_OFF: double (nullable = true)
 |-- WHEELS_ON: double (nullable = true)
 |-- TAXI_IN: double (nullable = true)
 |-- CRS_ARR_TIME: double (nullable = true)
 |-- ARR_TIME: double (nullable = true)
 |-- ARR_DELAY: double (nullable = true)
 |-- CANCELLED: double (nullable = true)
 |-- CANCELLATION_CODE: string (nullable = true)
 |-- DIVERTED: double (nullable = true)
 |-- CRS_ELAPSED_TIME: double (nullable = true)
 |-- ACTUAL_ELAPSED_TIME: double (nullable = true)
 |-- AIR_TIME: double (nullable = true)
 |-- DISTANCE: double (nullable = true)
 |-- CARRIER_DELAY: double (nullable = true)
 |-- WEATHER_DELAY: double

In [5]:
# For simplicity, let's use these basic features
selected_features = ['DEP_DELAY', 'DISTANCE', 'AIR_TIME', 'MONTH', 'DAY_OF_WEEK']

# Target variable - let's predict if flight is delayed (ARR_DELAY > 15 minutes)
df = df.withColumn("DELAYED", (df.ARR_DELAY > 15).cast("integer"))

# Select only the columns we need
model_data = df.select(selected_features + ['DELAYED'])

# Drop any rows with missing values
model_data = model_data.na.drop()

In [6]:
# Assemble features into a vector
assembler = VectorAssembler(
    inputCols=selected_features,
    outputCol="features"
)

# Split data into train and test sets
train_data, test_data = model_data.randomSplit([0.7, 0.3], seed=42)

In [7]:
# Initialize logistic regression
lr = LogisticRegression(featuresCol='features', labelCol='DELAYED')

# Create pipeline
pipeline = Pipeline(stages=[assembler, lr])

# Train model
model = pipeline.fit(train_data)

In [8]:
# Make predictions on test data
predictions = model.transform(test_data)

# Evaluate model performance
from pyspark.ml.evaluation import BinaryClassificationEvaluator

evaluator = BinaryClassificationEvaluator(labelCol="DELAYED")
accuracy = evaluator.evaluate(predictions)

print(f"Model AUC: {accuracy}")

Model AUC: 0.7063237647622311


In [9]:
from pyspark.ml.classification import RandomForestClassifier

# Random Forest model
rf = RandomForestClassifier(
    featuresCol='features',
    labelCol='DELAYED',
    numTrees=50,
    maxDepth=5,
    seed=42
)

# Create pipeline
rf_pipeline = Pipeline(stages=[assembler, rf])

# Train model
rf_model = rf_pipeline.fit(train_data)

# Make predictions
rf_predictions = rf_model.transform(test_data)

# Evaluate
rf_accuracy = evaluator.evaluate(rf_predictions)
print(f"Random Forest AUC: {rf_accuracy}")

Random Forest AUC: 0.8121983837806409


In [10]:
from pyspark.ml.classification import GBTClassifier

# Gradient-Boosted Trees model
gbt = GBTClassifier(
    featuresCol='features',
    labelCol='DELAYED',
    maxIter=10,
    maxDepth=5,
    seed=42
)

# Create pipeline
gbt_pipeline = Pipeline(stages=[assembler, gbt])

# Train model
gbt_model = gbt_pipeline.fit(train_data)

# Make predictions
gbt_predictions = gbt_model.transform(test_data)

# Evaluate
gbt_accuracy = evaluator.evaluate(gbt_predictions)
print(f"GBT AUC: {gbt_accuracy}")

GBT AUC: 0.8205547722024009


In [11]:
# Compare all three models
print(f"Logistic Regression AUC: {accuracy}")
print(f"Random Forest AUC: {rf_accuracy}")
print(f"Gradient-Boosted Trees AUC: {gbt_accuracy}")

Logistic Regression AUC: 0.7063237647622311
Random Forest AUC: 0.8121983837806409
Gradient-Boosted Trees AUC: 0.8205547722024009


In [1]:
from pyspark.ml.evaluation import MulticlassClassificationEvaluator
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
def plot_confusion_matrix(predictions, model_name):
    # Convert to Pandas DataFrame for easier handling
    preds_and_labels = predictions.select("prediction", "DELAYED").toPandas()
    
    # Create confusion matrix
    confusion_matrix = pd.crosstab(
        preds_and_labels['DELAYED'], 
        preds_and_labels['prediction'], 
        rownames=['Actual'], 
        colnames=['Predicted']
    )
    
    # Plot
    plt.figure(figsize=(5, 5))
    sns.heatmap(confusion_matrix, annot=True, fmt='d', cmap='Blues')
    plt.title(f'Confusion Matrix - {model_name}')
    plt.show()
    
    # Calculate metrics
    evaluator = MulticlassClassificationEvaluator(
        labelCol="DELAYED", 
        predictionCol="prediction",
        metricName="accuracy"
    )
    accuracy = evaluator.evaluate(predictions)
    print(f"{model_name} Accuracy: {accuracy:.4f}")
    print("Confusion Matrix:")
    print(confusion_matrix)
    print("\n" + "="*50 + "\n")

In [3]:
# For Logistic Regression
plot_confusion_matrix(predictions, "Logistic Regression")

# For Random Forest
plot_confusion_matrix(rf_predictions, "Random Forest")

# For Gradient-Boosted Trees
plot_confusion_matrix(gbt_predictions, "Gradient-Boosted Trees")

NameError: name 'predictions' is not defined